In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression

# Load original dataset
df = pd.read_csv("crop_yield.csv")

# Separate features and target
X = df.drop("Yield_tons_per_hectare", axis=1)
y = df["Yield_tons_per_hectare"]

# Region column for grouped validation
groups = df["Region"]

# Define column types
categorical_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]
numerical_cols = ["Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest"]
binary_cols = ["Fertilizer_Used", "Irrigation_Used"]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("bin", "passthrough", binary_cols)
    ]
)

# Best-performing model
model = LinearRegression()

# Full pipeline
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# Group-based cross-validation
gkf = GroupKFold(n_splits=4)

results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    test_region = groups.iloc[test_idx].unique()[0]
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append((test_region, r2, rmse))
    
    print(f"Fold {fold+1}")
    print(f"Held-Out Region: {test_region}")
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print("-" * 40)

results_df = pd.DataFrame(results, columns=["Held-Out Region", "R2", "RMSE"])
print("\nFinal Regional Validation Results:")
print(results_df)

print("\nMean R²:", np.mean(results_df["R2"]))
print("Std R²:", np.std(results_df["R2"]))

Fold 1
Held-Out Region: North
R²: 0.9126
RMSE: 0.5015
----------------------------------------
Fold 2
Held-Out Region: West
R²: 0.9129
RMSE: 0.5003
----------------------------------------
Fold 3
Held-Out Region: South
R²: 0.9132
RMSE: 0.5001
----------------------------------------
Fold 4
Held-Out Region: East
R²: 0.9132
RMSE: 0.5000
----------------------------------------

Final Regional Validation Results:
  Held-Out Region        R2      RMSE
0           North  0.912608  0.501483
1            West  0.912946  0.500275
2           South  0.913157  0.500146
3            East  0.913189  0.500047

Mean R²: 0.9129748299334537
Std R²: 0.00023154886413735655
